# Rule-Based Java Decomposition Analysis

This notebook demonstrates the rule-based approach to detecting large Java classes that should be decomposed.

In [ ]:
# Import required libraries
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## 1. Load Configuration

In [ ]:
# Load configuration
with open('../config.yaml') as f:
    config = yaml.safe_load(f)

print("Project:", config['project']['name'])
print("\nThresholds:")
for metric, levels in config['rules']['thresholds'].items():
    print(f"  {metric}: {levels}")

## 2. Load and Explore Metrics Data

In [ ]:
# Load metrics data
metrics_df = pd.read_csv('../data/processed/code_metrics.csv')

print(f"Total classes analyzed: {len(metrics_df)}")
print(f"\nColumns: {list(metrics_df.columns)}")

metrics_df.head(10)

In [ ]:
# Summary statistics
metrics_df[['loc', 'nom', 'wmc', 'cbo', 'lcom', 'cc']].describe()

## 3. Visualize Metric Distributions

In [ ]:
# Plot metric distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

metrics_to_plot = ['loc', 'nom', 'wmc', 'cbo', 'lcom', 'cc']
thresholds = {
    'loc': 500,
    'nom': 20,
    'wmc': 60,
    'cbo': 15,
    'lcom': 0.7,
    'cc': 10
}

for ax, metric in zip(axes.flat, metrics_to_plot):
    ax.hist(metrics_df[metric], bins=20, edgecolor='black', alpha=0.7)
    ax.axvline(thresholds[metric], color='red', linestyle='--', 
               label=f'Threshold: {thresholds[metric]}')
    ax.set_xlabel(metric.upper())
    ax.set_ylabel('Frequency')
    ax.set_title(f'Distribution of {metric.upper()}')
    ax.legend()

plt.tight_layout()
plt.savefig('../results/charts/metric_distributions.png', dpi=150)
plt.show()

## 4. Apply Rule-Based Detection

In [ ]:
# Apply threshold rules
def classify_class(row):
    violations = []
    
    if row['loc'] > 500:
        violations.append('ExcessiveClassLength')
    if row['nom'] > 20:
        violations.append('TooManyMethods')
    if row['wmc'] > 60:
        violations.append('HighComplexity')
    if row['cbo'] > 15:
        violations.append('HighCoupling')
    if row['lcom'] > 0.7:
        violations.append('LowCohesion')
    
    return len(violations), ', '.join(violations)

metrics_df['violation_count'], metrics_df['violations'] = zip(*metrics_df.apply(classify_class, axis=1))
metrics_df['needs_decomposition'] = metrics_df['violation_count'] >= 2

print(f"Classes flagged for decomposition: {metrics_df['needs_decomposition'].sum()}")
print(f"Percentage: {metrics_df['needs_decomposition'].mean()*100:.1f}%")

In [ ]:
# Show flagged classes
flagged = metrics_df[metrics_df['needs_decomposition']].sort_values('violation_count', ascending=False)
flagged[['class_name', 'loc', 'nom', 'wmc', 'cbo', 'lcom', 'violation_count', 'violations']]

## 5. Correlation Analysis

In [ ]:
# Correlation heatmap
correlation_metrics = ['loc', 'nom', 'wmc', 'cbo', 'lcom', 'cc', 'nof', 'rfc']
corr_matrix = metrics_df[correlation_metrics].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Correlation Between Code Metrics')
plt.tight_layout()
plt.savefig('../results/charts/correlation_matrix.png', dpi=150)
plt.show()

## 6. Priority Scoring

In [ ]:
# Calculate priority scores
def calculate_priority(row):
    score = 0
    
    # LOC score (weight: 0.25)
    if row['loc'] > 1000:
        score += 5 * 0.25
    elif row['loc'] > 500:
        score += 3 * 0.25
    elif row['loc'] > 300:
        score += 1 * 0.25
    
    # WMC score (weight: 0.25)
    if row['wmc'] > 100:
        score += 5 * 0.25
    elif row['wmc'] > 60:
        score += 3 * 0.25
    elif row['wmc'] > 40:
        score += 1 * 0.25
    
    # CBO score (weight: 0.20)
    if row['cbo'] > 25:
        score += 5 * 0.20
    elif row['cbo'] > 15:
        score += 3 * 0.20
    elif row['cbo'] > 10:
        score += 1 * 0.20
    
    # LCOM score (weight: 0.15)
    if row['lcom'] > 0.9:
        score += 5 * 0.15
    elif row['lcom'] > 0.7:
        score += 3 * 0.15
    elif row['lcom'] > 0.5:
        score += 1 * 0.15
    
    # NOM score (weight: 0.15)
    if row['nom'] > 35:
        score += 5 * 0.15
    elif row['nom'] > 20:
        score += 3 * 0.15
    elif row['nom'] > 15:
        score += 1 * 0.15
    
    return round(score, 2)

metrics_df['priority_score'] = metrics_df.apply(calculate_priority, axis=1)

# Show top priority classes
top_priority = metrics_df.nlargest(10, 'priority_score')
top_priority[['class_name', 'loc', 'nom', 'wmc', 'cbo', 'lcom', 'priority_score', 'violations']]

## 7. Summary Report

In [ ]:
print("=" * 50)
print("RULE-BASED ANALYSIS SUMMARY")
print("=" * 50)
print(f"\nTotal classes analyzed: {len(metrics_df)}")
print(f"Classes flagged for decomposition: {metrics_df['needs_decomposition'].sum()}")
print(f"Percentage flagged: {metrics_df['needs_decomposition'].mean()*100:.1f}%")
print(f"\nAverage priority score: {metrics_df['priority_score'].mean():.2f}")
print(f"Max priority score: {metrics_df['priority_score'].max():.2f}")
print("\n" + "=" * 50)
print("TOP 5 CLASSES FOR DECOMPOSITION")
print("=" * 50)
for _, row in top_priority.head(5).iterrows():
    print(f"\n{row['class_name']}")
    print(f"  LOC: {row['loc']}, Methods: {row['nom']}, Priority: {row['priority_score']}")
    print(f"  Violations: {row['violations']}")